# Path Finding on Grid Graphs
This notebook explores various use cases of shortest path algorithms/ path finding algorithms. 
In particular, we will search for the shortest path on grid-like graphs,
 which are common in many applications such as pathfinding in games, image processing and others.

## Outline

* First, we we will create interesting grid graphs by generating mazes. 
  We randomly remove edges from a grid graph to create a maze-like structure.
* We derive a heuristic function to generate particular hard mazes.
* We use the generated mazes to inspect the behavior of various path finding algorithms, 
  such as breadth-first search (BFS), depth-first search (DFS), A* and their bidirectional variants. 
We will visualize the search process to gain insights into how these algorithms explore the graph.
* Next we will explore how to use path finding algorithms for image processing tasks:
    * We will use the shortest path algorithms on an 8-connected grid graph to navigate between obstacles in an image. 
    * We will use the shortest path algorithms to demonstrate how to implement an interactive image segmentation tool, 
    where the user can click on an image to select a start and end point, and the algorithm will find the shortest path between them while navigating around obstacles in the image.

In [ ]:
# lets import the libraries we need
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import skimage
from ipycanvas import Canvas, MultiCanvas, hold_canvas
from IPython.display import display
from ipywidgets import widgets
from ipywidgets import IntProgress
from ipywidgets import Button, HBox, VBox, Layout, Label, IntProgress, Dropdown,Output
from collections import deque
import time

# seed number for reproducibility
np.random.seed(42)

# Maze Generation and Visualization

We will start by implementing the necessary functions to create maze candidates (we will see soon why we call them candidates) and functions to visualize them:

* `grid_graph_edges_4nh` is a helper function that generates the edges of a grid graph with 4-neighborhood connectivity.
* `generate_maze_candidate` is a function that generates a maze by randomly removing edges from the grid graph.
* `draw_maze` is a function that visualizes the maze on a canvas, highlighting the start and end nodes.

In [ ]:
def grid_graph_edges_4nh(maze_shape):
    edges = []
    for x in range(maze_shape[0]):
        for y in range(maze_shape[1]):
            u = (x, y)
            if x + 1 < maze_shape[0]:
                v = (x + 1, y)
                yield (u, v)
            if y + 1 < maze_shape[1]:
                v = (x, y + 1)
                yield (u, v)

def generate_maze_candidate(maze_shape, p_edge):
    g = nx.Graph()

    for x in range(maze_shape[0]):
        for y in range(maze_shape[1]):
            g.add_node((x, y))
    for u, v in grid_graph_edges_4nh(maze_shape):
        if np.random.rand() < p_edge:
            g.add_edge(u, v)
    return g

def draw_maze(g, maze_shape, pixel_size, line_with=5, layout={'width': '30%'}, display_canvas=True, output=None, canvas=None, node_colors=None, start_node=None, end_node=None):
    if start_node is None:
        start_node = (0, 0)
    if end_node is None:
        end_node = (maze_shape[0] - 1, maze_shape[1] - 1)
    if canvas is None:
        canvas = Canvas(width=maze_shape[0] * pixel_size, height=maze_shape[1] * pixel_size, layout=layout)
    if display_canvas:
        display(canvas)

    with hold_canvas(canvas):

        canvas.line_width = line_with
        canvas.stroke_style = 'rgb(200, 200, 200)'
        
        for x in range(maze_shape[0]):
            for y in range(maze_shape[1]):
                cx = x * pixel_size
                cy = y * pixel_size
                if node_colors is not None:
                    canvas.fill_style = node_colors.get((x, y), 'white')
                else:
                    canvas.fill_style = 'white'
                canvas.fill_rect(cx, cy, pixel_size, pixel_size)
                canvas.stroke_rect(cx, cy, pixel_size, pixel_size)

        # draw the start and end nodes in a light blue color
        canvas.fill_style = 'lightblue'
        canvas.fill_rect(start_node[0] * pixel_size + line_with // 2, start_node[1] * pixel_size + line_with // 2, pixel_size - line_with, pixel_size - line_with)
        canvas.fill_rect(end_node[0] * pixel_size + line_with // 2, end_node[1] * pixel_size + line_with // 2, pixel_size - line_with, pixel_size - line_with)
        canvas.stroke_style = 'rgb(0, 0, 0)'
        
        for uv in grid_graph_edges_4nh(maze_shape):
            u, v = uv
            if output is not None:
                output.append(f"edge: {u} - {v}, exists: {g.has_edge(u, v)}")
            assert u in g.nodes, f"Node {u} is not in the graph"
            assert v in g.nodes, f"Node {v} is not in the graph"
            if not g.has_edge(u, v):

                x1, y1 = u
                x2, y2 = v
                
                # horizontal edge (will be seperated by a vertical line)
                if x1 != x2:
                    # vertical line!
                    yl0 = y1 * pixel_size
                    yl1 = yl0 + pixel_size
                    xl = (x1 + 1) * pixel_size
                    canvas.stroke_line(xl, yl0, xl, yl1)
                else:
                    # horizontal line
                    xl0 = x1 * pixel_size
                    xl1 = xl0 + pixel_size
                    yl = (y1 + 1) * pixel_size
                    canvas.stroke_line(xl0, yl, xl1, yl)
        
        # draw outline of the maze
        canvas.stroke_width = line_with
        canvas.stroke_style = 'rgba(0, 0, 0, 1)'
        canvas.stroke_rect(0, 0, maze_shape[0] * pixel_size, maze_shape[1] * pixel_size)

    return canvas



# Draw candidate mazes

Lets generate a maze candidate with an edge probability of 0.3 and visualize it:
We can see that the generated maze has a lot of edges removed, but there is a problem.
Lets assume we fix the start node to be (0, 0) and the end node to be (maze_shape[0] - 1, maze_shape[1] - 1), 
then there is a high probability that the start and end node are not connected at all,
which means that there is no path between them.
This is a problem for our path finding algorithms, because they will not be able to find a path between the start and end node.


In [ ]:
maze_shape = (20,20)
maze = generate_maze_candidate(maze_shape, p_edge=0.4)
draw_maze(maze, maze_shape, 50, line_with=5);

While a higher edge probability (e.g., 0.4) can increase the chances of the start and end nodes being connected, it does not guarantee it, and also it makes the maze less challenging for path finding algorithms, because there are more edges to explore.

In [ ]:
maze_shape = (20,20)
maze = generate_maze_candidate(maze_shape, p_edge=0.8)
draw_maze(maze, maze_shape, 50, line_with=5);

# Ensuring connectivity of start and end nodes

To ensure that the start and end nodes are connected, we just generate a random maze candidate and check if the start and end node are connected, if not we generate a new maze candidate until we find one that is connected.
When the edge probability is low, this can take a while, but it ensures that we have a valid maze to work with for our path finding algorithms. To speed up the process, we increase the edge
probability after a certain number of attempts, which increases the chances of finding a connected maze candidate.

 yet for si

In [ ]:
def generate_maze(maze_shape, p_edge, max_attempts=100, increment=0.01, silent=False):
    g = generate_maze_candidate(maze_shape, p_edge)
    # check if there is a path from the start node to the end node, if not, generate a new maze candidate until there is a path
    start_node = (0, 0)
    end_node = (maze_shape[0] - 1, maze_shape[1] - 1)
    attempts = 0
    found = False
    while not found:
        for _ in range(max_attempts):
            g = generate_maze_candidate(maze_shape, p_edge)
            if nx.has_path(g, start_node, end_node):
                found = True
                break
        if not found:
            p_edge = min(1, p_edge + increment)
            if not silent:
                print(f"No path found, increment p_edge to {p_edge:.2f} and trying again...")
    return g         
maze_shape = (20,20)
maze = generate_maze(maze_shape, 0.4)
draw_maze(maze, maze_shape, 50);

# Draw shortest path on maze

Since we have the machine to generate valid mazes, we can now use it to visualize the shortest path between the start and end node on the maze.
Here we will use the build in `shortest_path` function from `networkx` to find the shortest path between the start and end node, and then we will visualize it on the maze.
Since the graph is unweighted, we can use the default breadth-first search (BFS) algorithm to find the shortest path, which is guaranteed to find the shortest path in an unweighted graph.

In [ ]:
# lets try to find the shortest path from the start node to the end node using bfs
maze_shape = 20,20
maze = generate_maze(maze_shape, 0.5)
canvas = draw_maze(maze, maze_shape, 50)

start_node = (0, 0)
end_node = (maze_shape[0] - 1, maze_shape[1] - 1)
shortest_path = nx.shortest_path(maze, start_node, end_node)

def draw_path(canvas, path, pixel_size, line_width=5):
    with hold_canvas(canvas):
        canvas.stroke_style = 'red'
        canvas.line_width = line_width
        for i in range(len(path) - 1):
            x1, y1 = path[i]
            x2, y2 = path[i + 1]
            cx1 = x1 * pixel_size + pixel_size // 2
            cy1 = y1 * pixel_size + pixel_size // 2
            cx2 = x2 * pixel_size + pixel_size // 2
            cy2 = y2 * pixel_size + pixel_size // 2
            canvas.stroke_line(cx1, cy1, cx2, cy2)
draw_path(canvas, shortest_path, 50)


# The longest shortest path

To generate particularly hard mazes, we try the following:
We generate N random mazes and compute the shortest path between the start and end node for each maze. We store the length
of the shortest path for each maze and then we select the maze with the longest shortest path.
For comparison, we also store the maze with the shortest shortest path, which is the easiest maze to solve.
Note that this is an arbitary definition of a hard maze, but it is a simple one that allows us to generate mazes of varying difficulty.

In [ ]:
n_mazes = 200
progress = IntProgress(value=0, min=0, max=n_mazes, description='Progress:')
display(progress)

maze_shape = (30,30)
p_edge = 0.5
lengths = []
hardest_maze = None
easiest_maze = None
longest_length = 0
shortest_length = float('inf')
longest_shortest_path = None
shortest_shortest_path = None


for i in range(n_mazes):
    progress.value = i + 1
    maze = generate_maze(maze_shape, p_edge, silent=True)
    start_node = (0, 0)
    end_node = (maze_shape[0] - 1, maze_shape[1] - 1)
    shortest_path = nx.shortest_path(maze, start_node, end_node)
    length = len(shortest_path)
    lengths.append(length)
    if hardest_maze is None or length > longest_length:
        hardest_maze = maze.copy()
        longest_length = length
        longest_shortest_path = list(shortest_path)

    if easiest_maze is None or length < shortest_length:
        easiest_maze = maze.copy()
        shortest_length = length
        shortest_shortest_path = list(shortest_path)

# visualize the easiest and hardest maze with their shortest paths
canvas_easiest = draw_maze(easiest_maze, maze_shape, 50, layout={'width': '100%'}, display_canvas=False)
draw_path(canvas_easiest, shortest_shortest_path, 50)

canvas_hardest = draw_maze(hardest_maze, maze_shape, 50, layout={'width': '100%'}, display_canvas=False)
draw_path(canvas_hardest, longest_shortest_path, 50)    


label_easiest = Label(value=f'Easiest Maze (length={shortest_length})')
label_hardest = Label(value=f'Hardest Maze (length={longest_length})')
vbox_layout = Layout(width='45%', align_items='center')
vbox_easiest = VBox([label_easiest, canvas_easiest], layout=vbox_layout)
vbox_hardest = VBox([label_hardest, canvas_hardest], layout=vbox_layout)
display(HBox(
    [vbox_easiest, vbox_hardest], 
    layout=Layout(width='100%', justify_content='space-around')
))

# Distribution of shortest path lengths
We can also visualize the distribution of shortest path lengths for a large number of random mazes to get a sense of how the difficulty of the mazes varies with the edge probability.

In [ ]:
# display histogram of lengths
plt.hist(lengths, bins=20)
plt.xlabel('Length of shortest path')
plt.ylabel('Frequency')
plt.title('Histogram of Shortest Path Lengths')

# Visualize Path Finding Algorithms

Since we have a machine to generate mazes of varying difficulty, we can now use it to visualize the behavior of various path finding algorithms on these mazes.
We implement the following path finding algorithms:
* Breadth-First Search (BFS)
* Depth-First Search (DFS)
* A* Search
* Bidirectional BFS
* Bidirectional DFS

To have an interactive experience, we will visualize the search process of these algorithms on the easiest and hardest mazes that we generated earlier.
Clicking on nodes will change the start and end node for the path finding algorithms.
Clicking between a pair of adjacent nodes will toggle the edge between them, allowing us to modify the maze.

In [ ]:
class PathFindBase(object):
    def __init__(self, G, start, end):
        self.G = G
        self.start = start
        self.end = end
        self.predecessors = {start: None}
    
    def queues(self):
        raise NotImplementedError()
    def visited_nodes(self):
        raise NotImplementedError()


def visualize(maze_shape , maze, start_node, end_node, alg_classes, delay=0.01, pixel_size=50, line_width=5):

    canvas = draw_maze(maze, maze_shape, pixel_size, display_canvas=False, start_node=start_node, end_node=end_node)
    output = Output()

    # when clicking on a node, it becomes start, next click becomes end, next click will pick again start and so on,
    # when the click is between to pixels, we will toggle the edge (ie add or remove the edge) between the two pixels,
    flip_end = False

    @output.capture()
    def on_canvas_click(x, y):
        nonlocal flip_end, start_node, end_node

        # convert x,y to node coordinates
        node_x = x // pixel_size
        node_y = y // pixel_size
        node_center_x = node_x * pixel_size + pixel_size // 2
        node_center_y = node_y * pixel_size + pixel_size // 2

        click_dist_u = ((x - node_center_x) ** 2 + (y - node_center_y) ** 2) ** 0.5

        nh = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dx, dy in nh:
            neighbor_x = node_x + dx
            neighbor_y = node_y + dy
            neighbor_center_y = neighbor_y * pixel_size + pixel_size // 2
            
            if 0 <= neighbor_x < maze_shape[0] and 0 <= neighbor_y < maze_shape[1]:
                neighbor_center_x = neighbor_x * pixel_size + pixel_size // 2                
                click_dist_v = ((x - neighbor_center_x) ** 2 + (y - neighbor_center_y) ** 2) ** 0.5

                assert click_dist_u <= click_dist_v

                delta = click_dist_v - click_dist_u
                if delta < 10:
                    # toggle edge
                    u = (node_x, node_y)
                    v = (neighbor_x, neighbor_y)
                    if maze.has_edge(u, v):
                        maze.remove_edge(u, v)
                    else:
                        maze.add_edge(u, v)
                    draw_maze(maze, maze_shape, pixel_size, canvas=canvas, display_canvas=False, start_node=start_node, end_node=end_node)
                    return
        # if click is not close to any edge, we will toggle the start and end node
        if flip_end:

            # clear old end node
            canvas.fill_style = 'white'
            canvas.fill_rect(end_node[0] * pixel_size + line_width // 2, end_node[1] * pixel_size + line_width // 2, pixel_size - line_width, pixel_size - line_width)
            end_node = (node_x, node_y)
            canvas.fill_style = 'lightblue'
            canvas.fill_rect(node_x * pixel_size + line_width // 2, node_y * pixel_size + line_width // 2, pixel_size - line_width, pixel_size - line_width)
            flip_end = False

        else:            

            # clear old start node
            canvas.fill_style = 'white' 
            canvas.fill_rect(start_node[0] * pixel_size + line_width // 2, start_node[1] * pixel_size + line_width // 2, pixel_size - line_width, pixel_size - line_width)
            start_node = (node_x, node_y)
            canvas.fill_style = 'lightblue'
            canvas.fill_rect(node_x * pixel_size + line_width // 2, node_y * pixel_size + line_width // 2, pixel_size - line_width, pixel_size - line_width)
            flip_end = True

    
    
    canvas.on_mouse_down(on_canvas_click)

    def run(button, alg_cls):
        nonlocal start_node, end_node
        draw_maze(maze, maze_shape, pixel_size, canvas=canvas, display_canvas=False, start_node=start_node, end_node=end_node)
        # disable button while running
        button.disabled = True
        alg = alg_cls(maze, start_node, end_node)
        node_colors = dict()
        for _ in alg.run():
            if delay is not None and delay > 0:
                time.sleep(delay)
            with hold_canvas(canvas):
                # highlight visited nodes
                for visited in alg.visited_nodes():
                    for node in visited:
                        node_colors[node] = 'rgb(255, 200, 200)'

                for q in alg.queues():

                    # highlight queue nodes
                    for node in q:
                        x,y = node
                        node_colors[node] = 'rgb(200, 200, 255)'
                    
                # # redraw maze
                # draw_maze(maze, maze_shape, pixel_size, display_canvas=False, canvas=canvas, node_colors=node_colors)
                node_colors[end_node] = 'lightblue'
                node_colors[start_node] = 'lightblue'
                for node, color in node_colors.items():
                    x,y = node
                    cx = x * pixel_size + line_width // 2
                    cy = y * pixel_size + line_width // 2
                    canvas.fill_style = color
                    canvas.fill_rect(cx, cy, pixel_size - line_width, pixel_size - line_width)
                
        
        # Reconstruct and draw the shortest path
        path = []
        current = end_node
        while current is not None:
            path.append(current)
            current = alg.predecessors.get(current)
        path.reverse()
        draw_path(canvas, path, 50)
        # enable button after running
        button.disabled = False

    # a list of algorithms to run as dropdown
    alg_dropdown = Dropdown(options=[alg_cls.__name__ for alg_cls in alg_classes], description='Algorithm:')

    def get_current_alg_cls():
        selected_alg_name = alg_dropdown.value
        for alg_cls in alg_classes:
            if alg_cls.__name__ == selected_alg_name:
                return alg_cls
        return None

    run_button = Button(description='Run')
    run_button.on_click(lambda b: run(run_button, alg_cls=get_current_alg_cls()))


    display(HBox([run_button, alg_dropdown], layout=Layout(width='100%', justify_content='flex-start')))
    display(canvas) 
    display(output)


With the code above to run and visualize different pathfinding algorithms, we can now implement the BFS and DFS algorithms as subclasses of PathFindBase and visualize their behavior on the same maze.

In [ ]:
maze = hardest_maze

# step wise bfs with predecessor tracking
from collections import deque
import time



class BFS(PathFindBase):
    def __init__(self, G, start, end):
        super().__init__(G, start, end) 
        self.queue = deque([start])
        self.visited = set([start])
        
    def queues(self):
        yield self.queue
    
    def visited_nodes(self):
        yield self.visited
    
    def run(self):
        while self.queue:
            current = self.queue.popleft()
            yield
            if current == self.end:
                break

            for neighbor in self.G.neighbors(current):
                if neighbor not in self.visited:
                    self.visited.add(neighbor)
                    self.queue.append(neighbor)
                    self.predecessors[neighbor] = current

# Depth-First Search
class DFS(PathFindBase):
    def __init__(self, G, start, end):
        super().__init__(G, start, end)
        self.stack = [start]
        self.visited = set([start])
    
    def queues(self):
        yield self.stack
    
    def visited_nodes(self):
        yield self.visited
    
    def run(self):
        while self.stack:
            current = self.stack.pop()
            yield
            if current == self.end:
                break
            
            for neighbor in self.G.neighbors(current):
                if neighbor not in self.visited:
                    self.visited.add(neighbor)
                    self.stack.append(neighbor)
                    self.predecessors[neighbor] = current




class BidirectionalBFS(PathFindBase):
    def __init__(self, G, start, end):
        super().__init__(G, start, end) 
        self.queue_start = deque([start])
        self.queue_end = deque([end])
        self.visited_front = set([start])
        self.visited_end = set([end])
        self.predecessors_end = {end: None}
    
    def queues(self):
        yield self.queue_start
        yield self.queue_end
    
    def visited_nodes(self):
        yield self.visited_front
        yield self.visited_end

    def run(self):
        meeting_node = None

        while self.queue_start and self.queue_end:
            # Expand from the start side
            current_start = self.queue_start.popleft()
            if current_start in self.visited_end:
                meeting_node = current_start
                break
            yield
            
            for neighbor in self.G.neighbors(current_start):
                if neighbor not in self.visited_front:
                    self.visited_front.add(neighbor)
                    self.queue_start.append(neighbor)
                    self.predecessors[neighbor] = current_start

            # Expand from the end side
            current_end = self.queue_end.popleft()
            if current_end in self.visited_front:
                meeting_node = current_end
                break
            yield

            for neighbor in self.G.neighbors(current_end):
                if neighbor not in self.visited_end:
                    self.visited_end.add(neighbor)
                    self.queue_end.append(neighbor)
                    self.predecessors_end[neighbor] = current_end

        # RECONSTRUCTION LOGIC
        if meeting_node is not None:
            # The 'start' side is already in self.predecessors correctly.
            # We need to flip the 'end' side links to point towards the start.
            curr = meeting_node
            while curr in self.predecessors_end:
                nxt = self.predecessors_end[curr]
                if nxt is not None:
                    self.predecessors[nxt] = curr
                curr = nxt


import heapq

# step wise A* with predecessor tracking
class AStar(PathFindBase):
    def __init__(self, G, start, end):
        super().__init__(G, start, end)
        self.visited = set([start])
        self.counter = 0
        self.heap = [(0, self.counter, start)]
        self.g_scores = {start: 0}
        self.heap_nodes = []
        
    def heuristic(self, node):
        return abs(node[0] - self.end[0]) + abs(node[1] - self.end[1])
    
    def queues(self):
        yield self.heap_nodes
    
    def visited_nodes(self):
        yield self.visited
    
    def run(self):
        while self.heap:
            f_score, _, current = heapq.heappop(self.heap)
            self.heap_nodes = [node for _, _, node in self.heap]
            yield
            
            if current == self.end:
                break
            
            for neighbor in self.G.neighbors(current):
                tentative_g_score = self.g_scores[current] + 1
                
                if neighbor not in self.g_scores or tentative_g_score < self.g_scores[neighbor]:
                    self.g_scores[neighbor] = tentative_g_score
                    f_score = tentative_g_score + self.heuristic(neighbor)
                    self.predecessors[neighbor] = current
                    
                    if neighbor not in self.visited:
                        self.visited.add(neighbor)
                        self.counter += 1
                        heapq.heappush(self.heap, (f_score, self.counter, neighbor))


# Bidirectional Depth-First Search
class BidirectionalDFS(PathFindBase):
    def __init__(self, G, start, end):
        super().__init__(G, start, end) 
        self.stack_start = [start]
        self.stack_end = [end]
        self.visited_front = set([start])
        self.visited_end = set([end])
        self.predecessors_end = {end: None}
    
    def queues(self):
        yield self.stack_start
        yield self.stack_end
    
    def visited_nodes(self):
        yield self.visited_front
        yield self.visited_end

    def run(self):
        meeting_node = None

        while self.stack_start and self.stack_end:
            # Expand from the start side
            current_start = self.stack_start.pop()
            if current_start in self.visited_end:
                meeting_node = current_start
                break
            yield
            
            for neighbor in self.G.neighbors(current_start):
                if neighbor not in self.visited_front:
                    self.visited_front.add(neighbor)
                    self.stack_start.append(neighbor)
                    self.predecessors[neighbor] = current_start

            # Expand from the end side
            current_end = self.stack_end.pop()
            if current_end in self.visited_front:
                meeting_node = current_end
                break
            yield

            for neighbor in self.G.neighbors(current_end):
                if neighbor not in self.visited_end:
                    self.visited_end.add(neighbor)
                    self.stack_end.append(neighbor)
                    self.predecessors_end[neighbor] = current_end

        # RECONSTRUCTION LOGIC
        if meeting_node is not None:
            curr = meeting_node
            while curr in self.predecessors_end:
                nxt = self.predecessors_end[curr]
                if nxt is not None:
                    self.predecessors[nxt] = curr
                curr = nxt

In [ ]:

start_node = (0, 0)
end_node = (maze_shape[0] - 1, maze_shape[1] - 1)
visualize(maze_shape, hardest_maze, start_node, end_node, [BFS, BidirectionalBFS, DFS, BidirectionalDFS, AStar], delay=0.01)


# Path Finding for Image Processing

In the last part of this notebook, we will explore how to use path finding algorithms for image processing tasks.
In particular, we will use the shortest path algorithms on an 8-connected grid graph to navigate between obstacles in an image. 

We start by implementing a function to generated an 8-connected **weighted** grid graph.

In [ ]:
# create graph with 8-connected pixels
def eight_connected_grid_graph(shape, edge_weight_fn):
    
    def yield_edges():
        for x in range(shape[0]):
            for y in range(shape[1]):
                for dx in (-1, 0, 1):
                    for dy in (-1, 0, 1):
                        if dx == 0 and dy == 0:
                            continue
                        nx_, ny_ = x + dx, y + dy
                        if 0 <= nx_ < shape[0] and 0 <= ny_ < shape[1]:
                            if nx_ > x or (nx_ == x and ny_ > y):
                                yield (x, y), (nx_, ny_), edge_weight_fn(x, y, nx_, ny_)
    g = nx.Graph()
    for u, v, w in yield_edges():
        g.add_edge(u, v, weight=w)
    return g

## Avoid obstacles by finding the shortest path in a graph
As an example, we will use the shortest path algorithms to navigate between obstacles in an image.
We will use the `coins` image from `skimage.data` as an example image, which contains several coins on a dark background.
We will try to "navigate" between the coins by finding the shortest path between two points in the image, while avoiding the coins as obstacles.

In [ ]:
# sample image
img = skimage.data.coins()
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

To create the edge weights such that we avoid the coins, we use some image processing technique to detect the edges of the coins.
To make the crossing of edges particular expensive, we take the exponential of the edge detection result.

In [ ]:
#  use gradient magnitude as edge weights
smoothed = skimage.filters.gaussian(img, sigma=2)
edges = skimage.filters.sobel(smoothed)
edges = edges / np.max(edges)

# use some exponential to make the weights more pronounced
weights = np.exp(edges * 4)


# plot height map
plt.imshow(weights, cmap='gray')
plt.axis('off')
plt.show()

To convert the weight-image into edge weights, we do the following:
For an edge between two adjacent pixels, we take the **maximum** of the values at the two pixels..

In [ ]:
# take max
def edge_weight_fn(x1, y1, x2, y2):
    return max(weights[x1, y1], weights[x2, y2]) + 0.5
g = eight_connected_grid_graph(weights.shape[0:2], edge_weight_fn)

# Interactive Path Finding

Finally, we show the interactively show the shortest path between two points in the image after each pair of clicks.


In [ ]:
# interactive ipycanvas widget to allow for clicking on the graph
from ipycanvas import Canvas, hold_canvas

canvas = Canvas(width=weights.shape[1], height=weights.shape[0])
points = []
draw_img = img

def draw_point(x, y, color='red', r=3):
    canvas.fill_style = color
    canvas.fill_circle(x, y, r)

def on_click(x, y):
    global points
    node = (int(y), int(x))
    if len(points) == 2:
        points = []
        canvas.put_image_data(draw_img, 0, 0)
    points.append(node)

    draw_point(x, y, 'red' if len(points) == 1 else 'blue')

    if len(points) == 2:
        path = nx.shortest_path(g, source=points[0], target=points[1], weight='weight')
        with hold_canvas(canvas):
            for i in range(len(path) - 1):
                y1, x1 = path[i]
                y2, x2 = path[i + 1]
                canvas.stroke_style = 'green'
                canvas.line_width = 2
                canvas.stroke_line(x1, y1, x2, y2)

canvas.on_mouse_down(on_click)
canvas.put_image_data(img, 0, 0)
display(canvas)

# Interactive Image Segmentation with Shortest Path Algorithms

Finally, we show how to use the shortest path algorithms to implement an interactive image segmentation tool, where the user can click on an image to select a start and end point, and the algorithm will find the shortest path between them while navigating around obstacles in the image.
In that case, we like the shortest path to follow the edges of the image.

In [ ]:

# load image (img)
img = skimage.data.astronaut()

smoothed = skimage.filters.gaussian(img, sigma=2)

# add channel wise sobel filter responses to get an edge map
sobel_r = skimage.filters.sobel(smoothed[:, :, 0])
sobel_g = skimage.filters.sobel(smoothed[:, :, 1]) 
sobel_b = skimage.filters.sobel(smoothed[:, :, 2])

edge_map = sobel_r + sobel_g + sobel_b

# show image and edge_map side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img)
axes[0].set_title('Original Image')
axes[0].axis('off')
axes[1].imshow(edge_map, cmap='gray')
axes[1].set_title('Edge Map (Sobel Magnitude)')
axes[1].axis('off')
plt.tight_layout()
plt.show()



to convert the height map into edge weights, we use the following formula:
`edge_weight = exp(-max(weights[x1, y1], weights[x2, y2]) * 15.0) + 0.5`
The constant part we add is to add a slight bias towards straight paths.

Click the image to create a start point, then click another point to create an end point, and the shortest path between them will be shown.
Hold the mouse button and drag to create a path with multiple control points, and the shortest path will be updated in real time as you move the mouse.
When a path is closed (i.e., the last point is close to the first point), the path will be closed and a new path can be started.

In [ ]:

# output widget 
from ipywidgets import Output
out = Output()
display(out)
# create graph and edge weight function for the image
def edge_weight_fn(x1, y1, x2, y2):

    y1, x1 = x1, y1
    y2, x2 = x2, y2
    
    max_gradient = max(edge_map[x1, y1], edge_map[x2, y2])
    # a high gradient should have a low weight.
    # so lets use some negative exponential to make the weights more pronounced
    return np.exp(-max_gradient * 15.0) + 0.5

graph = eight_connected_grid_graph(edge_map.shape[0:2], edge_weight_fn)

# interactive canvas for segmentation
multi_canvas = MultiCanvas(3, width=edge_map.shape[1], height=edge_map.shape[0])
canvas_image = multi_canvas[0]
canvas_stored_seg = multi_canvas[1]
canvas_temp = multi_canvas[2]
seg_points = []
seg_path = []
draw_img = img


canvas_image.put_image_data(draw_img, 0, 0)

def draw_point_seg(canvas, x, y, color='red', r=3 ):
    canvas.fill_style = color
    canvas.fill_circle(x, y, r)

def draw_line_seg(canvas, x1, y1, x2, y2, color='green', width=2):
    canvas.stroke_style = color
    canvas.line_width = width
    canvas.stroke_line(x1, y1, x2, y2)

is_down = False
ctrl_points = []


def draw_shortest_path(canvas, start, end):
    path = nx.shortest_path(graph, source=start, target=end, weight='weight')
    with hold_canvas(canvas):
        for i in range(len(path) - 1):
            x1, y1 = path[i]
            x2, y2 = path[i + 1]
            draw_line_seg(canvas, x1, y1, x2, y2)

last_x, last_y = 0, 0
@out.capture()
def on_mouse_move(x, y):
    x, y = int(x), int(y)
    try:
        global is_down, ctrl_points, last_x, last_y
        if is_down and len(ctrl_points) > 0 and (x, y) != (last_x, last_y):
            # clear temp canvas
            canvas_temp.clear()

            # draw shortest path from last ctrl point to current mouse position
            draw_shortest_path(canvas_temp, ctrl_points[-1], (x, y))

            last_x, last_y = x, y
    except Exception as e:
        print(f"Error in on_mouse_move: {e}")
@out.capture()
def on_mouse_down(x, y):
    x, y = int(x), int(y)
    try:
        global is_down, ctrl_points
        is_down = True

        # snap to start point if close enough and there are at least 3 ctrl points already
        if len(ctrl_points) >= 3:
            start_point = ctrl_points[0]
            dist_to_start = ((x - start_point[0]) ** 2 + (y - start_point[1]) ** 2) ** 0.5
            if dist_to_start < 10:
                x, y = start_point

        # clear temp canvas
        canvas_temp.clear()

        # draw point
        draw_point_seg(canvas_temp, x, y)

        # draw shortest path from last ctrl point to current mouse position if there is a ctrl point already
        if len(ctrl_points) > 0:
            draw_shortest_path(canvas_temp, ctrl_points[-1], (x, y))

    except Exception as e:
        print(f"Error in on_mouse_down: {e}")
        

@out.capture()
def on_mouse_up(x, y):
    x, y = int(x), int(y)
    try:
        global is_down, ctrl_points
        closed_segment = False

        n_points = len(ctrl_points)
        if n_points >= 3:
            # if the distance between this point and the start point is less than 3 pixels, we will consider this as closing the loop and we will connect this point to the start point and clear the ctrl points
            start_point = ctrl_points[0]
            dist_to_start = ((x - start_point[0]) ** 2 + (y - start_point[1]) ** 2) ** 0.5
            if dist_to_start < 10:
                x, y = start_point
                closed_segment = True

        is_down = False
        ctrl_points.append((x, y))
        # draw point to permanent canvas
        draw_point_seg(canvas_stored_seg, x, y)

        # draw shortest path from last ctrl point to current mouse position if there are 2 ctrl points already
        if len(ctrl_points) >= 2:   
            draw_shortest_path(canvas_stored_seg, ctrl_points[-2], ctrl_points[-1])
        if closed_segment:
            draw_shortest_path(canvas_stored_seg, ctrl_points[-1], ctrl_points[0])
            ctrl_points = []
    except Exception as e:
        print(f"Error in on_mouse_up: {e}")


@out.capture()
def on_mouse_out(x, y):
    try:
        global is_down
        is_down = False
    except Exception as e:
        print(f"Error in on_mouse_out: {e}")

canvas_temp.on_mouse_down(on_mouse_down)
canvas_temp.on_mouse_move(on_mouse_move)
canvas_temp.on_mouse_up(on_mouse_up)
canvas_temp.on_mouse_out(on_mouse_out)
display(multi_canvas)